# elevator-rmnd

Use the simulated dataset generated by `dataset.py` to train
regression models, in order to predict the remaining useful life (RUL)
of lifts.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

## preprocessing
* Import the simulated dataset
* Convert lift ids to integers
* ~~Compute instantaneous time derivatives of each sensor metric~~ [called off for now]
* Compute RUL in h
* Split data into training/testing sets

In [2]:
df_full = pd.read_csv("predictive_maintenance_lifts.csv")
df_full.head(10)

,timestamp,lift_id,lift_model,lift_age_hours,ARM_DIST_mm,DOOR_DIST_mm,FLOOR_DIST_mm,ROPE_MFL_mV,BEARING_TEMP_C,maintenance_done
0,2023-01-01 00:00:00,LIFT_001,Otis Gen2,15000,0.506,2.101,0.116,9.737,34.429,False
1,2023-01-01 12:00:00,LIFT_001,Otis Gen2,15012,0.490,1.883,0.386,10.544,33.781,False
2,2023-01-02 00:00:00,LIFT_001,Otis Gen2,15024,0.509,2.214,0.252,9.202,34.980,False
3,2023-01-02 12:00:00,LIFT_001,Otis Gen2,15036,0.539,2.688,0.668,10.468,35.398,False
4,2023-01-03 00:00:00,LIFT_001,Otis Gen2,15048,0.524,2.671,0.844,10.225,35.197,False
5,2023-01-03 12:00:00,LIFT_001,Otis Gen2,15060,0.541,2.667,0.981,10.564,36.912,False
6,2023-01-04 00:00:00,LIFT_001,Otis Gen2,15072,0.528,2.974,1.734,11.412,37.046,False
7,2023-01-04 12:00:00,LIFT_001,Otis Gen2,15084,0.580,2.908,1.058,11.337,36.593,False
8,2023-01-05 00:00:00,LIFT_001,Otis Gen2,15096,0.544,3.065,1.345,11.153,38.018,False
9,2023-01-05 12:00:00,LIFT_001,Otis Gen2,15108,0.598,3.386,1.919,11.118,37.117,False


Convert `lift_model` to integer labels

In [3]:
# Convert unique string lift IDs to integer indices
lift_models = df_full["lift_model"].unique()
df_full["lift_idx"] = df_full["lift_model"].apply(
    lambda x: np.where(lift_models == x)[0][0]
)
df_full.head(30)

,timestamp,lift_id,lift_model,lift_age_hours,ARM_DIST_mm,DOOR_DIST_mm,FLOOR_DIST_mm,ROPE_MFL_mV,BEARING_TEMP_C,maintenance_done,lift_idx
0,2023-01-01 00:00:00,LIFT_001,Otis Gen2,15000,0.506,2.101,0.116,9.737,34.429,False,0
1,2023-01-01 12:00:00,LIFT_001,Otis Gen2,15012,0.490,1.883,0.386,10.544,33.781,False,0
2,2023-01-02 00:00:00,LIFT_001,Otis Gen2,15024,0.509,2.214,0.252,9.202,34.980,False,0
3,2023-01-02 12:00:00,LIFT_001,Otis Gen2,15036,0.539,2.688,0.668,10.468,35.398,False,0
4,2023-01-03 00:00:00,LIFT_001,Otis Gen2,15048,0.524,2.671,0.844,10.225,35.197,False,0
5,2023-01-03 12:00:00,LIFT_001,Otis Gen2,15060,0.541,2.667,0.981,10.564,36.912,False,0
6,2023-01-04 00:00:00,LIFT_001,Otis Gen2,15072,0.528,2.974,1.734,11.412,37.046,False,0
7,2023-01-04 12:00:00,LIFT_001,Otis Gen2,15084,0.580,2.908,1.058,11.337,36.593,False,0
8,2023-01-05 00:00:00,LIFT_001,Otis Gen2,15096,0.544,3.065,1.345,11.153,38.018,False,0
9,2023-01-05 12:00:00,LIFT_001,Otis Gen2,15108,0.598,3.386,1.919,11.118,37.117,False,0


Compute the **RUL** for each record in the dataset by
calculating the number of hours between the current log
and the next maintenance event (where `maintenance_done == 1`).

This computation should be _specific_ to the lift model only, i.e. filter by lift model first.

In [ ]:
# Compute the RUL for each record in the dataset
def compute_rul(record: pd.Series, avail_maintenances: pd.DataFrame) -> float:
    """
    Compute the remaining useful life (RUL) in h for a given record.
    """
    time_now = record["timestamp"]
    # Get the next available maintenance (should be greater than the current time)
    next_available_maintenance = avail_maintenances[
        avail_maintenances["timestamp"] >= time_now
    ]["timestamp"].min()
    # If no further maintenance exists, then RUL is np.inf
    if pd.isna(next_available_maintenance):
        return np.inf
    # If there exists a further maintenance, compute the RUL in hours
    time_now = pd.to_datetime(time_now).to_pydatetime()
    next_available_maintenance = pd.to_datetime(next_available_maintenance).to_pydatetime()
    return (next_available_maintenance - time_now).total_seconds() / 3600

maintenance_events = df_full[df_full["maintenance_done"] == 1]
for model in lift_models:
    # Get a list of all maintenance events for the current lift model
    available_maintenance_events = maintenance_events[maintenance_events["lift_model"] == model]
    # Get all log events for the current lift model
    log_events = df_full[df_full["lift_model"] == model]
    # Then compute the RUL for each log event for each lift model
    for idx, record in log_events.iterrows():
        rul = compute_rul(record, available_maintenance_events)
        df_full.at[idx, "RUL_hrs"] = rul
assert not any(df_full["RUL_hrs"].isna()), "RUL_hrs should not have any null values after computation"
assert not any(df_full["RUL_hrs"] < 0), "RUL_hrs should not have any negative values after computation"
df_full.tail(10)

,timestamp,lift_id,lift_model,lift_age_hours,ARM_DIST_mm,DOOR_DIST_mm,FLOOR_DIST_mm,ROPE_MFL_mV,BEARING_TEMP_C,maintenance_done,lift_idx,RUL_hrs
5990,2025-09-22 00:00:00,LIFT_003,KONE MonoSpace,32380,0.538,2.248,0.404,10.401,35.210,False,2,inf
5991,2025-09-22 12:00:00,LIFT_003,KONE MonoSpace,32392,0.515,2.477,0.553,10.795,37.511,False,2,inf
5992,2025-09-23 00:00:00,LIFT_003,KONE MonoSpace,32404,0.574,2.504,0.811,11.066,36.981,False,2,inf
5993,2025-09-23 12:00:00,LIFT_003,KONE MonoSpace,32416,0.558,2.698,1.278,10.917,38.318,False,2,inf
5994,2025-09-24 00:00:00,LIFT_003,KONE MonoSpace,32428,0.561,2.824,0.767,11.587,39.209,False,2,inf
5995,2025-09-24 12:00:00,LIFT_003,KONE MonoSpace,32440,0.572,2.928,1.234,10.961,38.791,False,2,inf
5996,2025-09-25 00:00:00,LIFT_003,KONE MonoSpace,32452,0.563,3.185,1.417,11.930,39.436,False,2,inf
5997,2025-09-25 12:00:00,LIFT_003,KONE MonoSpace,32464,0.593,3.153,1.761,12.002,40.189,False,2,inf
5998,2025-09-26 00:00:00,LIFT_003,KONE MonoSpace,32476,0.584,3.333,1.602,12.211,41.024,False,2,inf
5999,2025-09-26 12:00:00,LIFT_003,KONE MonoSpace,32488,0.592,3.640,1.715,11.641,42.279,False,2,inf


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score

Since there exist rows that have `np.inf` as their predicted RUL, extract those rows for experimentation later on.
Retain the remaining roles (with valid RULs) for training and testing.

In [11]:
df_experimental = df_full[df_full["RUL_hrs"] == np.inf]  # Filter out records with no further maintenance
df_useful = df_full[df_full["RUL_hrs"] != np.inf]  # Retain the remaining records for training/testing

# Check to see the experimental/useful split was done correctly
assert all(df_experimental["RUL_hrs"] == np.inf), "All records in experimental set should have RUL_hrs as np.inf"
assert all(df_useful["RUL_hrs"] != np.inf), "All records in useful set should have valid RUL_hrs"

df_useful.head(40)

,timestamp,lift_id,lift_model,lift_age_hours,ARM_DIST_mm,DOOR_DIST_mm,FLOOR_DIST_mm,ROPE_MFL_mV,BEARING_TEMP_C,maintenance_done,lift_idx,RUL_hrs
0,2023-01-01 00:00:00,LIFT_001,Otis Gen2,15000,0.506,2.101,0.116,9.737,34.429,False,0,504.0
1,2023-01-01 12:00:00,LIFT_001,Otis Gen2,15012,0.490,1.883,0.386,10.544,33.781,False,0,492.0
2,2023-01-02 00:00:00,LIFT_001,Otis Gen2,15024,0.509,2.214,0.252,9.202,34.980,False,0,480.0
3,2023-01-02 12:00:00,LIFT_001,Otis Gen2,15036,0.539,2.688,0.668,10.468,35.398,False,0,468.0
4,2023-01-03 00:00:00,LIFT_001,Otis Gen2,15048,0.524,2.671,0.844,10.225,35.197,False,0,456.0
5,2023-01-03 12:00:00,LIFT_001,Otis Gen2,15060,0.541,2.667,0.981,10.564,36.912,False,0,444.0
6,2023-01-04 00:00:00,LIFT_001,Otis Gen2,15072,0.528,2.974,1.734,11.412,37.046,False,0,432.0
7,2023-01-04 12:00:00,LIFT_001,Otis Gen2,15084,0.580,2.908,1.058,11.337,36.593,False,0,420.0
8,2023-01-05 00:00:00,LIFT_001,Otis Gen2,15096,0.544,3.065,1.345,11.153,38.018,False,0,408.0
9,2023-01-05 12:00:00,LIFT_001,Otis Gen2,15108,0.598,3.386,1.919,11.118,37.117,False,0,396.0


Then perform a 80/20 `train_test_split` on the remaining useful data.

In [14]:
X = df_useful.drop(
    columns=["timestamp", "lift_id", "lift_model", "maintenance_done", "RUL_hrs"]
)
y = df_useful["RUL_hrs"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=67
)

In [15]:
# Dummy checks
assert len(X_train) == len(y_train), "X_train and y_train should have the same number of records"
assert len(X_test) == len(y_test), "X_test and y_test should have the same number of records"
assert y_train.isna().sum() == 0, "y_train should not have any null values"
assert y_test.isna().sum() == 0, "y_test should not have any null values"

In [18]:
print(y_train.isna().sum())
print(y_test.isna().sum())

0
0


## model implementation 1: `SVR`
Use [`SVR`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html) to regress $Y$ (RUL) against $X$ (all metadata and sensor readings).
Support vector regression was chosen because it has the ability to handle non-linear relationships.

In [19]:
from sklearn.svm import SVR

In [ ]:
svr = SVR(kernel="linear", epsilon=0.2)
svr.fit(X_train, y_train)
y_pred = svr.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"MSE (SVR): {mse:.4f}")
acc = accuracy_score(y_test, y_pred.round())
print(f"Accuracy (SVR): {acc:.4f}")